# Experiment 3 – Counterfactual Swap

Goal: test whether SHAP's attribution to an identity word matches the word's counterfactual effect (the measured change in logit when the identity word is swapped for a neutral control word), re-test the Exp 2 'debiasing' share result, and measure how much SHAP changes with the removal baseline. Runtime: Colab → Runtime → Change runtime type → T4 GPU.

Spec: `experiments/counterfactual swap/EXP3_counterfactual_swap.md`. Expected runtime about 45–60 minutes (mostly Cells 6 and 9).

## Cell 1: install, imports and shared helpers (copied from Exp 1, plus Exp 3 additions)

In [ ]:
!pip -q install detoxify shap lime
import importlib.metadata, os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, shap, lime, sklearn
from matplotlib import colors as mcolors
from detoxify import Detoxify
from lime.lime_text import LimeTextExplainer
from scipy.stats import spearmanr
print("shap", shap.__version__, "| lime", importlib.metadata.version("lime"), "| torch", torch.__version__)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
OUT = "outputs_exp3"; os.makedirs(OUT, exist_ok=True)
BG, FG = "#FFFFFF", "#000000"                     # pink theme (same as Exp 2): white background, black text, pink bars
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
                     "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
                     "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
                     "xtick.color": FG, "ytick.color": FG, "legend.labelcolor": FG})
COLORS = {"original": "#D6006F", "unbiased": "#FF8FC4", "err": "#333333"}   # side-by-side pair differs strongly in lightness
ATTR_CMAP = mcolors.LinearSegmentedColormap.from_list("grey_white_pink", ["#7A7A7A", "#FFFFFF", "#D6006F"])   # away from toxic = grey, toward = pink

MODELS = {m: Detoxify(m, device=DEVICE) for m in ["original", "unbiased"]}

def proba(texts, model="original", bs=64):
    """P(toxic) for a list of strings, batched so LIME's thousands of samples don't run out of memory."""
    texts = [str(t) for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        out.extend(np.atleast_1d(MODELS[model].predict(texts[i:i+bs])["toxicity"]))
    return np.array(out, dtype=float)

def logit(texts, model="original"):
    """Explain log-odds, not probability, so SHAP contributions are additive (shap-exercises notebook)."""
    p = np.clip(proba(texts, model), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def lime_fn(model="original"):
    """friday-03 pattern: list of strings -> (n, 2) array of [P(non-toxic), P(toxic)]."""
    def f(texts):
        p = proba(texts, model)
        return np.column_stack([1 - p, p])
    return f

def word_tokenizer(s, return_offsets_mapping=True):
    """Word-level tokens that keep their trailing punctuation. shap's built-in regex tokenizer drops the
    sentence-final full stop, so SHAP would explain a different string than the one we score."""
    spans = [m.span() for m in re.finditer(r"\w+\W*|\W+", s)]
    out = {"input_ids": [s[a:b] for a, b in spans]}
    if return_offsets_mapping: out["offset_mapping"] = spans
    return out
MASKER = shap.maskers.Text(word_tokenizer)    # word-level tokens so SHAP and LIME use the same units
def shap_explainer(model="original"):
    return shap.Explainer(lambda t: logit(t, model), MASKER)

def check_efficiency(sv, texts, model, tol=1e-2):
    """Efficiency/additivity axiom (W4-L01 p4, textbook p48): base value + sum of SHAP values = model output."""
    f = logit(texts, model)
    recon = np.array([float(sv[i].base_values) + sv[i].values.sum() for i in range(len(texts))])
    err = np.abs(recon - f).max()
    assert err < tol, f"Efficiency check FAILED, max error {err}"
    print(f"[{model}] efficiency check passed, max error = {err:.2e}")

def shap_word_weights(e):
    """Collapse SHAP tokens into lower-case words; sum repeated words."""
    w = {}
    for tok, v in zip(e.data, e.values):
        k = re.sub(r"\W+", "", tok).lower()
        if k: w[k] = w.get(k, 0.0) + float(v)
    return w

def lime_word_weights(text, model="original", seed=SEED, num_samples=2000):
    """LIME on one sentence. labels=(1,) so weights refer to the TOXIC class (friday-01 sign trap)."""
    expl = LimeTextExplainer(class_names=["non-toxic", "toxic"], random_state=seed)
    n = len(set(re.findall(r"\w+", text)))
    exp = expl.explain_instance(text, lime_fn(model), num_features=n,
                                num_samples=num_samples, labels=(1,))
    return {w.lower(): v for w, v in exp.as_list(label=1)}, exp

def agreement(sw, lw, k=3):
    """Krishna et al. (2022)-style agreement: rank correlation, top-1 match, top-k overlap.
    Ranks are used because SHAP is in log-odds and LIME is in probability, so raw sizes are not comparable."""
    common = sorted(set(sw) & set(lw))
    rho = spearmanr([sw[w] for w in common], [lw[w] for w in common]).correlation if len(common) > 2 else np.nan
    top = lambda d, n: [w for w in sorted(d, key=lambda w: -d[w])[:n]]
    return dict(rho=rho, top1_agree=top(sw, 1) == top(lw, 1),
                top3_overlap=len(set(top(sw, k)) & set(top(lw, k))) / k)

def highlight_png(pairs, title, path):
    """Draw a sentence with each word coloured by its attribution (pink = toward toxic, grey = away).
    Colours are scaled per figure, so compare colour patterns between SHAP and LIME, not exact shades."""
    vmax = max([abs(v) for _, v in pairs] + [1e-9])
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax); cmap = ATTR_CMAP
    fig, ax = plt.subplots(figsize=(14, 1.4)); ax.axis("off")
    fig.canvas.draw(); r = fig.canvas.get_renderer(); x = 0.005
    for w, v in pairs:
        t = ax.text(x, 0.5, w, fontsize=20, va="center", transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.25", fc=cmap(norm(v)), ec="none"))
        x = t.get_window_extent(renderer=r).transformed(ax.transAxes.inverted()).x1 + 0.01
    ax.set_title(title, loc="left", fontsize=16)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)

# ---- Exp 3 additions ----
from scipy.stats import pearsonr
print("default mask token:", repr(MASKER.mask_token))   # we expect '...'; write down what it prints

# Colour scheme: same pink-and-black theme as the Exp 1 and Exp 2 figures (values from 02_identity_swap.ipynb)
COLORS.update({"control": "#9A8590", "n500": "#A3134F", "n5000": "#FF7AB8",
               "SHAP": "#D6006F", "LIME": "#FF8FC4", "random": "#9A8590"})
PINK_CMAP = mcolors.LinearSegmentedColormap.from_list("pink_black", ["#FFF0F6", "#FF4FA3", "#7A0A3E"])

def make_masker(mask_token):
    """Same word-level tokens as MASKER, different replacement for a 'removed' word."""
    return shap.maskers.Text(word_tokenizer, mask_token=mask_token)

def shap_explainer(model="original", masker=None):
    return shap.Explainer(lambda t: logit(t, model), masker or MASKER)

def faith(ref, est):
    """Faithfulness of an estimate to a reference: correlation, sign agreement, mean absolute error."""
    ref, est = np.asarray(ref, float), np.asarray(est, float)
    return dict(pearson=pearsonr(ref, est)[0], spearman=spearmanr(ref, est).correlation,
                sign_agree=float(np.mean(np.sign(ref) == np.sign(est))),
                mae=float(np.mean(np.abs(ref - est))))

## Cell 2: the Exp 2 grid (do not edit)

In [ ]:
TEMPLATES = ["I am a {} person.", "My neighbour is {} and she is lovely.",
             "{} people are welcome here.", "My best friend is {}.",
             "I grew up in a {} family.", "Being {} is completely normal.",
             "We hired a {} engineer last week.", "She is a proud {} woman.",
             "He is {} and works as a teacher.", "Our new {} classmate joined the team."]
TERMS = ["gay","lesbian","transgender","black","white","asian","muslim","jewish",
         "christian","immigrant","deaf","blind","disabled","poor","rural"]
CONTROLS = ["tall","young","local","quiet"]
grid = pd.DataFrame([(t, w, w in CONTROLS, t.format(w)) for t in TEMPLATES for w in TERMS + CONTROLS],
                    columns=["template","term","is_control","text"])
print(len(grid), "sentences")   # expect 190
grid.to_csv(f"{OUT}/exp3_sentences.csv", index=False)

## Cell 3: scores

In [ ]:
for m in MODELS:
    grid[f"p_{m}"] = proba(grid.text, m)
    grid[f"logit_{m}"] = logit(grid.text, m)
print(grid.groupby("is_control")[["p_original", "p_unbiased"]].mean().round(4))

## Cell 4: SHAP with the fixed tokenizer (default masker)

In [ ]:
SV = {}
for m in MODELS:
    SV[m] = shap_explainer(m)(list(grid.text), max_evals=500, batch_size=32)
    check_efficiency(SV[m], list(grid.text), m)
    ww = [shap_word_weights(SV[m][i]) for i in range(len(grid))]
    grid[f"phi_term_{m}"] = [ww[i].get(grid.term[i], np.nan) for i in range(len(grid))]
    grid[f"phi_share_{m}"] = [abs(grid[f"phi_term_{m}"][i]) / max(np.abs(SV[m][i].values).sum(), 1e-9)
                              for i in range(len(grid))]
    grid[f"base_{m}"] = [float(SV[m][i].base_values) for i in range(len(grid))]
assert grid[[f"phi_term_{m}" for m in MODELS]].notna().all().all(), "a term was not found in SHAP's tokens"
print(grid.groupby("is_control")[[f"phi_term_{m}" for m in MODELS]].mean().round(3))

## Cell 5: counterfactual effect vs SHAP estimate (the headline measure)

For each identity sentence:
- `cf` = its logit − the mean logit of the 4 control sentences in the same template (**the reference**).
- `est` = its identity word's SHAP value − the mean control-word SHAP value in the same template (**SHAP's estimate**).
- `spill` = `cf − est` (how much of the real effect SHAP put on other words).

In [ ]:
for m in MODELS:
    c = grid[grid.is_control].groupby("template")
    grid[f"cf_{m}"]    = grid[f"logit_{m}"]    - grid.template.map(c[f"logit_{m}"].mean())
    grid[f"est_{m}"]   = grid[f"phi_term_{m}"] - grid.template.map(c[f"phi_term_{m}"].mean())
    grid[f"spill_{m}"] = grid[f"cf_{m}"] - grid[f"est_{m}"]
ids = grid[~grid.is_control].copy()

rows = []
for m in MODELS:
    r = faith(ids[f"cf_{m}"], ids[f"est_{m}"]); r.update(model=m, level="sentence (n=150)"); rows.append(r)
    t = ids.groupby("term")[[f"cf_{m}", f"est_{m}"]].mean()
    r = faith(t[f"cf_{m}"], t[f"est_{m}"]); r.update(model=m, level="term (n=15)"); rows.append(r)
faith_shap = pd.DataFrame(rows); print(faith_shap.round(3))
faith_shap.to_csv(f"{OUT}/exp3_faithfulness_shap.csv", index=False)

## Cell 6: the same check for LIME

LIME weights are in probability units, so compare them with the counterfactual effect **in probability** and use rank correlation only. One LIME run per sentence: seed 42, 5000 samples (LIME's default, W5-L02 p2).

In [ ]:
for m in MODELS:
    lw = [lime_word_weights(t, m, seed=SEED, num_samples=5000)[0] for t in grid.text]
    grid[f"lime_term_{m}"] = [lw[i].get(grid.term[i], np.nan) for i in range(len(grid))]
    c = grid[grid.is_control].groupby("template")
    grid[f"cf_p_{m}"]     = grid[f"p_{m}"]         - grid.template.map(c[f"p_{m}"].mean())
    grid[f"est_lime_{m}"] = grid[f"lime_term_{m}"] - grid.template.map(c[f"lime_term_{m}"].mean())
    print(f"[{m}] LIME done")
ids = grid[~grid.is_control].copy()
faith_lime = pd.DataFrame([dict(model=m, **faith(ids[f"cf_p_{m}"], ids[f"est_lime_{m}"])) for m in MODELS])
print(faith_lime[["model", "spearman", "sign_agree"]].round(3))
faith_lime.to_csv(f"{OUT}/exp3_faithfulness_lime.csv", index=False)
grid.to_csv(f"{OUT}/exp3_results.csv", index=False)      # Exp 6 needs this file

## Cell 7: Figure 1 (paper Fig. 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4), sharex=True, sharey=True)
for ax, m in zip(axes, MODELS):
    ax.scatter(ids[f"cf_{m}"], ids[f"est_{m}"], s=12, alpha=0.7, color=COLORS[m])
    lim = [min(ids[f"cf_{m}"].min(), ids[f"est_{m}"].min()) - 0.5,
           max(ids[f"cf_{m}"].max(), ids[f"est_{m}"].max()) + 0.5]
    ax.plot(lim, lim, "--", color=FG, lw=0.8)
    for term, g in ids.groupby("term"):
        ax.annotate(term, (g[f"cf_{m}"].mean(), g[f"est_{m}"].mean()), fontsize=7)
    r = faith(ids[f"cf_{m}"], ids[f"est_{m}"])
    ax.set_title(f"Detoxify {m}  (ρ = {r['spearman']:.2f})", fontsize=10)
    ax.set_xlabel("Counterfactual effect (log-odds)", fontsize=9)
    ax.tick_params(labelsize=8); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("SHAP estimate (log-odds)", fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_exp3_cf_vs_shap.pdf"); fig.savefig(f"{OUT}/fig_exp3_cf_vs_shap.png", dpi=300)
plt.show()

## Cell 8: re-test the "46% → 60%" claim

In [ ]:
rt = pd.DataFrame({m: {"share_of_|SHAP| (presentation measure)": ids[f"phi_share_{m}"].mean(),
                       "mean phi_term (log-odds)": ids[f"phi_term_{m}"].mean(),
                       "mean SHAP estimate (log-odds)": ids[f"est_{m}"].mean(),
                       "mean counterfactual effect (log-odds)": ids[f"cf_{m}"].mean()} for m in MODELS})
print(rt.round(3)); rt.to_csv(f"{OUT}/exp3_debias_retest.csv")
per_term = ids.groupby("term")[[f"{k}_{m}" for m in MODELS for k in ("phi_share", "phi_term", "cf")]].mean()
per_term.round(3).to_csv(f"{OUT}/exp3_debias_retest_per_term.csv")
print(per_term.round(3))

*If the share went up but the absolute SHAP value and the counterfactual effect went down, the presentation's "46% → 60%" was a side-effect of the share measure, not a sign that the unbiased model leans on the word more. Record which it is in the journal.*

## Cell 9: baseline check (how much does the removal baseline matter?)

Three ways of "removing" a word: `"..."` (shap's default, already done in Cell 4, and what Exp 1 and Exp 2 used), deletion, and each model's own mask token (`[MASK]` for original, `<mask>` for unbiased).

Two shap quirks (tested on shap 0.52):
- `mask_token=""` crashes shap's text masker, so **deletion uses a single space `" "`**. The removed word just disappears ("My best friend gay.").
- Mask tokens get **a trailing space** (`"[MASK] "`), otherwise they glue onto the next word ("[MASK]gay."), which changes how RoBERTa tokenizes that word.

The first lines print one masked sentence per baseline so you can check the masking does what you think.

In [ ]:
BASELINES = {m: {"dots": "...", "delete": " ", "model_mask": MODELS[m].tokenizer.mask_token + " "} for m in MODELS}
print(BASELINES)
demo = grid.text[0]; n_tok = len(word_tokenizer(demo)["input_ids"])
demo_mask = np.array([True] * (n_tok - 2) + [False, True])     # remove the second-to-last word
for m in MODELS:
    for name, tok in BASELINES[m].items():
        print(m, name, "->", make_masker(tok)(demo_mask, demo))

bl_rows, term_rank = [], {}
for m in MODELS:
    for name, tok in BASELINES[m].items():
        if name == "dots":
            phi = grid[f"phi_term_{m}"]
        else:
            sv = shap_explainer(m, make_masker(tok))(list(grid.text), max_evals=500, batch_size=32)
            check_efficiency(sv, list(grid.text), m)
            ww = [shap_word_weights(sv[i]) for i in range(len(grid))]
            phi = pd.Series([ww[i].get(grid.term[i], np.nan) for i in range(len(grid))], index=grid.index)
            grid[f"phi_term_{m}_{name}"] = phi
        est = phi - grid.template.map(phi[grid.is_control].groupby(grid.template).mean())
        r = faith(grid.loc[~grid.is_control, f"cf_{m}"], est[~grid.is_control])
        r.update(model=m, baseline=name, control_mean_abs_phi=float(phi[grid.is_control].abs().mean()))
        bl_rows.append(r)
        term_rank[(m, name)] = phi[~grid.is_control].groupby(grid.term).mean()
bl = pd.DataFrame(bl_rows); print(bl.round(3)); bl.to_csv(f"{OUT}/exp3_baselines.csv", index=False)
rank_rows = []
for m in MODELS:   # do the three baselines rank the 15 identity terms the same way?
    for a, b in [("dots", "delete"), ("dots", "model_mask"), ("delete", "model_mask")]:
        rho = spearmanr(term_rank[(m, a)], term_rank[(m, b)]).correlation
        rank_rows.append(dict(model=m, a=a, b=b, term_ranking_rho=rho))
        print(m, a, "vs", b, "term-ranking rho =", round(rho, 3))
pd.DataFrame(rank_rows).to_csv(f"{OUT}/exp3_baseline_rankings.csv", index=False)
grid.to_csv(f"{OUT}/exp3_results.csv", index=False)      # re-save with the baseline columns

## Cell 10: results summary (numbers to quote in the paper)

In [ ]:
print("=== EXPERIMENT 3 SUMMARY ===")
print("default mask token:", repr(MASKER.mask_token))
for m in MODELS:
    s = faith(ids[f"cf_{m}"], ids[f"est_{m}"])
    print(f"[{m}] SHAP vs counterfactual effect: Spearman {s['spearman']:.2f}, Pearson {s['pearson']:.2f}, "
          f"sign agreement {s['sign_agree']:.0%}, MAE {s['mae']:.2f} log-odds (n=150)")
    print(f"[{m}] mean spill-over onto other words: {ids[f'spill_{m}'].mean():.2f} log-odds")
    print(f"[{m}] LIME vs counterfactual effect (prob.): Spearman "
          f"{faith(ids[f'cf_p_{m}'], ids[f'est_lime_{m}'])['spearman']:.2f}")
    print(f"[{m}] control words: mean |phi| = {grid.loc[grid.is_control, f'phi_term_{m}'].abs().mean():.3f} log-odds")
print("Weakest-faithfulness terms (largest mean |spill|, original):")
print(ids.groupby("term")["spill_original"].apply(lambda s: s.abs().mean()).sort_values(ascending=False).head(5).round(2))
print(bl[["model", "baseline", "spearman", "mae", "control_mean_abs_phi"]].round(3))

## Cell 11: download everything

In [ ]:
!zip -qr outputs_exp3.zip outputs_exp3
from google.colab import files; files.download("outputs_exp3.zip")

## After running

1. Unzip the outputs into `experiments/counterfactual swap/results/`.
2. Go through the checks in Part 6 of the spec (efficiency passed, control words near 0, P(toxic) matches Exp 2, masked examples look right).
3. Copy the Cell 10 summary into a journal entry, including the default mask token and whether the 46% → 60% claim survived.
4. Commit under your own GitHub account: `research: experiment 3 counterfactual swap results`.
5. Tell whoever runs Exp 6 that `results/exp3_results.csv` is pushed.